# ARC-Jeni-2026 — Hybrid DSA + Program Synthesis + Chakra-Ensemble LLM

**Competition:** ARC Prize 2026 – ARC-AGI-2  
**Author:** joyjeni  
**Repo:** https://github.com/joyjeni/arc-agi2-hybrid-solver  

## Architecture (3-tier)
| Tier | Method | Time Budget | Expected Coverage |
|------|--------|-------------|-------------------|
| 1 | DSA + Vedic deterministic solvers | 300 s total / 5 s task | 5–10% |
| 2 | LLM Program Synthesis (write+run Python) | 40% remaining / 45 s task | 15–25% |
| 3 | Chakra-8 Ensemble direct prediction | remainder / 90 s task | 10–20% |

**Model:** Qwen3-4B fine-tuned on ARC grids (`qwen3_4b_grids15_sft139`)  
**Hard time control:** Tier-1 ThreadPoolExecutor wall-clock; Tier-2/3 `signal.SIGALRM` per task  
**Total wall-clock target:** ≤5 h (18 000 s)

---
### v33 fixes (validated locally on the 120 public ARC-AGI-2 evaluation tasks)
1. **Model discovery** — `MODEL_DIR` is found by searching `/kaggle/input` for the Qwen `config.json`
   instead of one hard-coded path. If the model isn't attached, the log now says so clearly.
2. **Honest tier counts** — the nearest-neighbour fallback was labelled tier 1, so the log reported
   "Tier-1 (DSA): 240" even when no rule solver fired. It is now tier 0 and counted as fallback.
3. **Removed the in-notebook `kaggle competitions submit` call** — it fails with internet off and
   raises `FileNotFoundError` wherever the CLI is missing. Submit from the Kaggle UI instead.

**Known limitation (not fixable by a patch):** the Tier-1 solvers solve 320/1000 public *training*
tasks (the ones they were written from) but 0/120 unseen *evaluation* tasks, and the NN fallback
cannot be correct by construction. Expect a leaderboard score near 0 unless the LLM tiers solve tasks.


In [ ]:
# ── Dependency check (internet DISABLED — all packages pre-installed on Kaggle) ─
import yaml  # pyyaml
print('pyyaml OK:', yaml.__version__)


In [ ]:
# ── Setup & Imports ─────────────────────────────────────────────────────────
import os, sys, time, json, logging, signal, subprocess
try:
    import kagglehub  # standard Kaggle notebook boilerplate
except ImportError:
    kagglehub = None  # not available outside Kaggle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, wait as _fut_wait

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('arc_jeni')

IS_KAGGLE = Path('/kaggle').exists()
IS_RERUN  = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

# Competition data: try primary path first, then fallback (matches Robina V19 pattern)
_KAGGLE_COMP_PRIMARY  = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
_KAGGLE_COMP_FALLBACK = Path('/kaggle/input/arc-prize-2026-arc-agi-2')
if IS_KAGGLE:
    DATA_DIR = _KAGGLE_COMP_PRIMARY if _KAGGLE_COMP_PRIMARY.exists() else _KAGGLE_COMP_FALLBACK
else:
    DATA_DIR = Path('data')
def _find_model_dir():
    """Locate the attached Qwen model by its config.json (mount paths vary by Kaggle version)."""
    if not IS_KAGGLE:
        return None
    preferred = Path('/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1')
    if (preferred / 'config.json').exists():
        return preferred
    for cfg in sorted(Path('/kaggle/input').glob('**/config.json')):
        p = str(cfg.parent).lower()
        if 'qwen' in p or 'grids15' in p:
            return cfg.parent
    return None

MODEL_DIR  = _find_model_dir()
if IS_KAGGLE and MODEL_DIR is None:
    print('WARNING: no Qwen model found under /kaggle/input — attach the model or LLM tiers will be skipped')
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.monotonic()   # competition start timestamp

print(f'IS_KAGGLE={IS_KAGGLE}  IS_RERUN={IS_RERUN}')
print(f'DATA_DIR={DATA_DIR}  MODEL_DIR={MODEL_DIR}')
# ── Budget constants & time helper ──────────────────────────────────────────
TOTAL_BUDGET_S    = 9 * 3600 - 120   # 9h Kaggle limit minus 2-min safety margin
COMPETITION_LIMIT_S = 9 * 3600          # full 9-hour competition window
T1_BUDGET_S       = 300               # tier-1 DSA pass budget (5 min)
T1_PER_TASK_S     = 3.0               # per-task limit for tier-1
T2_SHARE          = 0.50              # fraction of remaining budget for tier-2
T2_MAX_BUDGET_S   = 4 * 3600         # hard cap for tier-2 total
T2_MAX_PER_TASK_S = 120              # max 2 min per task in tier-2
T2_MIN_PER_TASK_S = 5                # abort tier-2 if fair share drops below 5s
T3_MAX_PER_TASK_S = 60               # max 1 min per task in tier-3
T3_MIN_PER_TASK_S = 10               # abort tier-3 if fair share drops below 10s

def _remaining() -> float:
    """Seconds remaining in the competition budget."""
    return max(0.0, TOTAL_BUDGET_S - (time.monotonic() - T0))


In [ ]:
# Cell 4: Embed source files from zlib+base64 bundle
import json, zlib, base64, os, sys, textwrap

BUNDLE_B64 = (
    "..."
    "YnvdtR+1VqVznBZ/turR/uN3Rs9iRA=="
)

_raw = zlib.decompress(base64.b64decode(BUNDLE_B64))
_bundle = json.loads(_raw.decode('utf-8'))

# Write files to /kaggle/working/src/
for rel_path, content in _bundle.items():
    abs_path = os.path.join('/kaggle/working', rel_path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    with open(abs_path, 'w', encoding='utf-8') as f:
        f.write(content)

# Add to sys.path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

print(f'Extracted {len(_bundle)} source files')


In [ ]:
# ── Load challenge data ──────────────────────────────────────────────────────
import os, glob as _glob
from pathlib import Path
from src.pipeline import load_challenges

# Diagnostic: show what is actually in /kaggle/input/
KAGGLE_INPUT = Path('/kaggle/input')
print('── /kaggle/input/ tree ──────────────────────────────────────────────')
for root, dirs, files in os.walk(KAGGLE_INPUT):
    depth = root.replace(str(KAGGLE_INPUT), '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{Path(root).name}/')
    sub = '  ' * (depth + 1)
    for f in files:
        print(f'{sub}{f}')
    if depth >= 3:
        dirs.clear()
print('────────────────────────────────────────────────────────────────────')

def _find_json(patterns):
    # 1) Direct path under known competition folder (most reliable, matches ref notebook)
    for pat in patterns:
        if '*' not in pat and '?' not in pat:
            direct = DATA_DIR / pat
            if direct.exists():
                return direct
    # 2) rglob fallback (handles renamed/nested layouts)
    for pat in patterns:
        hits = list(KAGGLE_INPUT.rglob(pat))
        if hits:
            return hits[0]
    return None

# ── Priority 1: test challenges (competition scoring) ────────────────────
CHALLENGES_FILE = _find_json(['arc-agi_test_challenges.json',
                               'arc-agi-test-challenges.json',
                               '*test*challenges*.json'])
if CHALLENGES_FILE is None:
    # Priority 2: evaluation challenges (dev/testing fallback)
    CHALLENGES_FILE = _find_json(['arc-agi_evaluation_challenges.json',
                                   'arc-agi-evaluation-challenges.json',
                                   '*evaluation*challenges*.json'])
    if CHALLENGES_FILE is None:
        all_json = list(KAGGLE_INPUT.rglob('*.json'))
        raise FileNotFoundError(
            'No test or evaluation challenges file found in /kaggle/input/\n'
            f'All JSON files present: {[str(p) for p in all_json]}'
        )
    logger.warning('Test challenges not found — using evaluation set as dev fallback')

print(f'Using challenges file: {CHALLENGES_FILE}')
tasks = load_challenges(CHALLENGES_FILE)
print(f'Loaded {len(tasks)} tasks from {CHALLENGES_FILE.name}')
print('Sample task IDs:', list(tasks.keys())[:5])
print(f'Remaining budget at data load: {_remaining()/3600:.2f}h')


In [ ]:
# ── Tier-0: DISABLED (training-solution lookup) ─────────────────────────────
# REASON: Kaggle competition serves DIFFERENT test inputs for the same task IDs.
# T0 returned training-solution outputs → 0.00 score because inputs are new.
# NN fallback (Cell 6) uses actual test inputs → scored 0.42 in Sep 2026.
# T0 is disabled; NN + T1 DSA handle all 240 tasks.
from src.core.interfaces import SolverResult, Prediction

def _load_json_safe(patterns):
    p = _find_json(patterns)
    if p is not None:
        try:
            data = json.loads(p.read_text())
            logger.info(f'Loaded {len(data)} entries from {p.name}')
            return data
        except Exception as _e:
            logger.warning(f'Failed to load {p}: {_e}')
    return None

# T0 disabled: NN fallback will cover all 240 tasks as the base tier
t0_results: dict = {}
solved_ids: set = set()

print(f'T0: DISABLED — all {len(tasks)} tasks forwarded to T1 DSA + NN fallback')
print(f'Remaining budget: {_remaining()/3600:.2f}h')


In [ ]:
# ── Tier-NN: Nearest-Neighbor fallback (pixel similarity on training pairs) ──
# For each test input: find training pair with most similar input → return its output.
# Fallback only: it copies a training output, so it is almost never correct on new tasks.
from src.core.interfaces import SolverResult, Prediction

def _grid_dist(g1, g2) -> int:
    """Pixel-level Hamming distance; inf if shapes differ."""
    if len(g1) != len(g2):
        return 10**9
    if g1 and g2 and len(g1[0]) != len(g2[0]):
        return 10**9
    return sum(g1[r][c] != g2[r][c]
               for r in range(len(g1))
               for c in range(len(g1[0])))

def _nn_predict(train_pairs, test_input):
    """Return output of nearest-neighbor training pair (by input pixel distance)."""
    same_sz = [p for p in train_pairs
               if len(p['input']) == len(test_input) and
               (not test_input or len(p['input'][0]) == len(test_input[0]))]
    pool = same_sz if same_sz else train_pairs
    best = min(pool, key=lambda p: _grid_dist(p['input'], test_input))
    return best['output']

def _nn_predict2(train_pairs, test_input):
    """Second-best NN prediction (next distinct output)."""
    same_sz = [p for p in train_pairs
               if len(p['input']) == len(test_input) and
               (not test_input or len(p['input'][0]) == len(test_input[0]))]
    pool = same_sz if same_sz else train_pairs
    sorted_pool = sorted(pool, key=lambda p: _grid_dist(p['input'], test_input))
    best_out = sorted_pool[0]['output']
    # Find next distinct output
    for p in sorted_pool[1:]:
        if p['output'] != best_out:
            return p['output']
    return best_out  # fallback: same as best

nn_results: dict = {}
for _tid, _ctx in tasks.items():
    _preds = []
    for _test_in in _ctx.test_inputs:
        _out1 = _nn_predict(_ctx.train_pairs, _test_in)
        _out2 = _nn_predict2(_ctx.train_pairs, _test_in)
        _preds.append(Prediction(
            attempt_1=_out1,
            attempt_2=_out2,
            solver_name='NN_Fallback',
            confidence=0.0,
        ))
    nn_results[_tid] = SolverResult(
        task_id=_tid, predictions=_preds, solved=False, tier=0)

print(f'NN: generated fallback predictions for {len(nn_results)}/{len(tasks)} tasks')
print(f'Remaining budget: {_remaining()/3600:.2f}h')


In [ ]:
# ── Tier-1: DSA + Vedic deterministic solvers (parallel) ────────────────────
from src.solvers import REGISTRY
from src.pipeline import run_tier1

print(f'Registered solvers ({len(REGISTRY.names())}): {REGISTRY.names()[:10]} ...')

t1_start = time.monotonic()
t1_results = run_tier1(
    tasks, solved_ids, REGISTRY,
    total_budget_s=T1_BUDGET_S,
    per_task_s=T1_PER_TASK_S,
)
t1_elapsed = time.monotonic() - t1_start
solved_ids |= set(t1_results.keys())
pct = 100 * len(solved_ids) / max(len(tasks), 1)
print(f'Tier-1 done in {t1_elapsed:.1f}s — solved {len(solved_ids)}/{len(tasks)} ({pct:.1f}%)')
print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── LLM Setup: Qwen3-4B from Kaggle model hub ───────────────────────────────
from src.llm.engine import build_engine

model_path = str(MODEL_DIR) if (MODEL_DIR and MODEL_DIR.exists()) else None
skip_llm_tiers = (model_path is None)
if skip_llm_tiers:
    logger.warning('Model not available — T2/T3 SKIPPED; fallback will copy test inputs for unsolved tasks')
    t2_results: dict = {}
    t3_results: dict = {}
else:
    logger.info(f'Model found at {model_path}')

llm_engine = build_engine(model_path=model_path)
print('LLM engine ready:', type(llm_engine).__name__, '| skip_llm_tiers:', skip_llm_tiers)

In [ ]:
# ── Per-task hard-interrupt handler (SIGALRM — Linux/Kaggle only) ───────────
# signal.SIGALRM only works on the main thread; notebooks run on main thread.

def _on_task_timeout(signum, frame):
    raise TimeoutError('Hard per-task timeout (SIGALRM)')

# Verify signal module is available
assert hasattr(signal, 'SIGALRM'), 'SIGALRM not available on this platform'
print('SIGALRM available — hard per-task timeout enabled')

In [ ]:
# ── Tier-2: Program Synthesis with hard per-task SIGALRM timeout ────────────
from src.llm.program_synth import ProgramSynthesisSolver

if skip_llm_tiers:
    print('Tier-2 skipped — no LLM model available')
else:

    t2_remaining_budget = min(T2_SHARE * _remaining(), T2_MAX_BUDGET_S)
    t2_pending = [(tid, ctx) for tid, ctx in tasks.items() if tid not in solved_ids]
    n_t2 = len(t2_pending)
    print(f'Tier-2: {n_t2} tasks | budget={t2_remaining_budget:.0f}s | max/task={T2_MAX_PER_TASK_S}s')

    t2_results: dict = {}
    t2_solver = ProgramSynthesisSolver(llm_engine)
    _old_t2_handler = signal.signal(signal.SIGALRM, _on_task_timeout)
    t2_loop_start = time.monotonic()

    for _i, (_tid, _ctx) in enumerate(t2_pending):
        _t2_elapsed = time.monotonic() - t2_loop_start
        _t2_left    = t2_remaining_budget - _t2_elapsed
        _tasks_left = n_t2 - _i
        if _t2_left < 30.0:
            logger.info(f'T2: budget exhausted at task {_i}/{n_t2} — stopping')
            break
        _fair_share = _t2_left / _tasks_left
        _per_task   = min(T2_MAX_PER_TASK_S, _fair_share)
        if _per_task < T2_MIN_PER_TASK_S:
            logger.info(f'T2: fair-share {_per_task:.1f}s < min {T2_MIN_PER_TASK_S}s — stopping')
            break
        # Set hard per-task alarm
        signal.alarm(max(1, int(_per_task)))
        try:
            _res = t2_solver._timed_solve(_ctx)
            signal.alarm(0)   # cancel alarm on success
            if _res and _res.solved:
                t2_results[_tid] = _res
        except TimeoutError:
            signal.alarm(0)
            logger.warning(f'T2: {_tid} hard-timeout after {_per_task:.1f}s')
        except Exception as _exc:
            signal.alarm(0)
            logger.debug(f'T2: {_tid} error: {_exc}')
        if (_i + 1) % 10 == 0:
            logger.info(f'T2 progress {_i+1}/{n_t2}: {len(t2_results)} solved | {_t2_left:.0f}s left')

    signal.signal(signal.SIGALRM, _old_t2_handler)  # restore
    solved_ids |= set(t2_results.keys())
    t2_elapsed_total = time.monotonic() - t2_loop_start
    print(f'Tier-2 done in {t2_elapsed_total:.1f}s — solved {len(t2_results)}/{n_t2} new tasks')
    print(f'Cumulative solved: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/max(len(tasks),1):.1f}%)')
    print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── Tier-3: Chakra Ensemble with hard per-task SIGALRM timeout ──────────────
from src.llm.chakra_ensemble import ChakraEnsembleSolver

if skip_llm_tiers:
    print('Tier-3 skipped — no LLM model available')
else:

    t3_budget = max(0.0, _remaining())
    t3_pending = [(tid, ctx) for tid, ctx in tasks.items() if tid not in solved_ids]
    n_t3 = len(t3_pending)
    print(f'Tier-3: {n_t3} tasks | budget={t3_budget:.0f}s ({t3_budget/3600:.2f}h) | max/task={T3_MAX_PER_TASK_S}s')

    t3_results: dict = {}
    _old_t3_handler = signal.signal(signal.SIGALRM, _on_task_timeout)
    t3_loop_start = time.monotonic()

    for _i, (_tid, _ctx) in enumerate(t3_pending):
        _t3_elapsed = time.monotonic() - t3_loop_start
        _t3_left    = t3_budget - _t3_elapsed
        _tasks_left = n_t3 - _i
        if _t3_left < 30.0:
            logger.info(f'T3: budget exhausted at task {_i}/{n_t3} — stopping')
            break
        _fair_share = _t3_left / _tasks_left
        _per_task   = min(T3_MAX_PER_TASK_S, _fair_share)
        if _per_task < T3_MIN_PER_TASK_S:
            logger.info(f'T3: fair-share {_per_task:.1f}s < min {T3_MIN_PER_TASK_S}s — stopping')
            break
        # Rebuild solver per task so per_task_timeout is always tight
        _solver = ChakraEnsembleSolver(llm_engine, n_templates=4, per_task_timeout=float(_per_task))
        signal.alarm(max(1, int(_per_task) + 5))  # +5s grace for thread cleanup
        try:
            _res = _solver._timed_solve(_ctx)
            signal.alarm(0)
            if _res and _res.solved:
                t3_results[_tid] = _res
        except TimeoutError:
            signal.alarm(0)
            logger.warning(f'T3: {_tid} hard-timeout after {_per_task:.1f}s')
        except Exception as _exc:
            signal.alarm(0)
            logger.debug(f'T3: {_tid} error: {_exc}')
        if (_i + 1) % 20 == 0:
            logger.info(f'T3 progress {_i+1}/{n_t3}: {len(t3_results)} solved | {_t3_left:.0f}s left')

    signal.signal(signal.SIGALRM, _old_t3_handler)  # restore
    solved_ids |= set(t3_results.keys())
    t3_elapsed_total = time.monotonic() - t3_loop_start
    print(f'Tier-3 done in {t3_elapsed_total:.1f}s — solved {len(t3_results)}/{n_t3} new tasks')
    print(f'Final solved: {len(solved_ids)}/{len(tasks)} ({100*len(solved_ids)/max(len(tasks),1):.1f}%)')
    print(f'Remaining budget: {_remaining()/3600:.2f}h')

In [ ]:
# ── Build & validate submission ─────────────────────────────────────────────
# Merge strategy (v32):
#   Priority 1 (highest): T2/T3 LLM override (when available)
#   Priority 2: T1 DSA solvers with NN as attempt_2
#   Priority 3: NN fallback — covers all tasks (tier 0; placeholder, rarely correct)
#   NOTE: T0 training-solution lookup REMOVED — it scored 0.00 by returning
#         answers for old training test inputs, not competition's actual inputs.
from src.core.submission import build_submission, validate_submission, write_submission

# Step 1: start with NN base (covers all 240 tasks)
all_results: dict = dict(nn_results)

# Step 2: T1 override with NN as attempt_2
for _tid, _t1res in t1_results.items():
    _nn = nn_results.get(_tid)
    if _nn:
        _merged_preds = []
        for _i, _t1p in enumerate(_t1res.predictions):
            _nnp = _nn.predictions[_i] if _i < len(_nn.predictions) else _t1p
            _merged_preds.append(Prediction(
                attempt_1=_t1p.attempt_1,
                attempt_2=_nnp.attempt_1,
                solver_name=_t1p.solver_name,
                confidence=_t1p.confidence,
            ))
        from src.core.interfaces import SolverResult as _SR
        all_results[_tid] = _SR(task_id=_tid, predictions=_merged_preds,
                                 solved=True, tier=1)
    else:
        all_results[_tid] = _t1res

# Step 3: T2/T3 full override (when LLM available)
for _tid, _res in {**t2_results, **t3_results}.items():
    all_results[_tid] = _res

# (Step 4: T0 removed — was overwriting correct NN answers with wrong training outputs)

print(f'Merge: NN={len(nn_results)} base | T1={len(t1_results)} | T2/T3={len(t2_results)+len(t3_results)}')

submission = build_submission(tasks, all_results, fallback_input=True)

errors = validate_submission(submission, tasks)
if errors:
    for e in errors[:10]:
        print('VALIDATION ERROR:', e)
    if len(errors) > 10:
        print(f'  ... and {len(errors)-10} more')
else:
    print(f'Submission valid: {len(submission)} tasks')

# Sample entry check
sample_tid = list(submission.keys())[0]
sample_pred = submission[sample_tid][0]
a1, a2 = sample_pred['attempt_1'], sample_pred['attempt_2']
print(f'Sample [{sample_tid}]: attempt_1 {len(a1)}x{len(a1[0])}  attempt_2 {len(a2)}x{len(a2[0])}')


In [ ]:
# ── Write submission.json ────────────────────────────────────────────────────
OUT_PATH = OUTPUT_DIR / 'submission.json'
write_submission(submission, OUT_PATH, validate=False)

# Verify written file
_loaded = json.loads(OUT_PATH.read_text())
assert len(_loaded) == len(tasks), f'Task count mismatch: {len(_loaded)} vs {len(tasks)}'
for _tid, _preds in _loaded.items():
    for _pred in _preds:
        assert 'attempt_1' in _pred and 'attempt_2' in _pred, f'Missing attempt keys in {_tid}'

total_elapsed = time.monotonic() - T0
_tier_counts = {
    1: sum(1 for r in all_results.values() if r.tier == 1),
    2: sum(1 for r in all_results.values() if r.tier == 2),
    3: sum(1 for r in all_results.values() if r.tier == 3),
}
_fallback = sum(1 for r in all_results.values() if r.tier == 0) + (len(tasks) - len(all_results))

print(f'submission.json written: {OUT_PATH}')
print(f'Total tasks : {len(tasks)}')
print(f'Tier-1 (DSA)       : {_tier_counts[1]}')
print(f'Tier-2 (ProgSynth) : {_tier_counts[2]}')
print(f'Tier-3 (Chakra LLM): {_tier_counts[3]}')
print(f'Fallback (NN / input copy, unsolved): {_fallback}')
print(f'Elapsed : {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f}h)')
print(f'Budget used: {100*total_elapsed/COMPETITION_LIMIT_S:.1f}%')

In [ ]:
# ── Verify submission.json ──────────────────────────────────────────────────
# Code competition: Kaggle scores the submission.json this notebook writes when you
# click "Submit" on a saved version. No CLI call is needed (internet is off anyway).
import json as _json
from pathlib import Path

_sub_path = OUTPUT_DIR / 'submission.json'
assert _sub_path.exists(), f'submission.json missing at {_sub_path}'
assert _sub_path.stat().st_size > 0, 'submission.json is empty!'

_sub_check = _json.loads(_sub_path.read_text())
print(f'submission.json verified: {len(_sub_check)} tasks  {_sub_path.stat().st_size/1024:.1f}KB')
print(f'Tiers: T1={_tier_counts[1]} T2={_tier_counts[2]} T3={_tier_counts[3]} '
      f'fallback={_fallback} | {total_elapsed/60:.1f} min')


In [ ]:
# ── Comprehensive Submission Validator ──────────────────────────────────────
import json, pathlib, sys

SUB_PATH = OUTPUT_DIR / 'submission.json'

# ── 1. File existence ────────────────────────────────────────────────────────
assert SUB_PATH.exists(), f"FAIL: submission.json not found at {SUB_PATH}"

raw = SUB_PATH.read_text()
assert len(raw) > 2, f"FAIL: submission.json is empty (size={len(raw)})"

sub = json.loads(raw)

# ── 2. Top-level structure ───────────────────────────────────────────────────
assert isinstance(sub, dict), f"FAIL: top-level must be dict, got {type(sub)}"
assert len(sub) > 0, "FAIL: submission is empty dict"

total_entries = 0
errors = []

for task_id, attempts in sub.items():
    # ── 3. Each value is a list ──────────────────────────────────────────────
    if not isinstance(attempts, list):
        errors.append(f"{task_id}: value must be list, got {type(attempts)}")
        continue
    if len(attempts) == 0:
        errors.append(f"{task_id}: empty list of attempts")
        continue

    for idx, entry in enumerate(attempts):
        total_entries += 1
        # ── 4. Each entry has exactly attempt_1 and attempt_2 ────────────────
        if not isinstance(entry, dict):
            errors.append(f"{task_id}[{idx}]: entry must be dict, got {type(entry)}")
            continue
        missing = {k for k in ('attempt_1', 'attempt_2') if k not in entry}
        extra   = set(entry.keys()) - {'attempt_1', 'attempt_2'}
        if missing:
            errors.append(f"{task_id}[{idx}]: missing keys {missing}")
        if extra:
            errors.append(f"{task_id}[{idx}]: unexpected keys {extra}")

        # ── 5. Validate each grid ────────────────────────────────────────────
        for key in ('attempt_1', 'attempt_2'):
            grid = entry.get(key)
            if grid is None:
                continue
            if not isinstance(grid, list) or len(grid) == 0:
                errors.append(f"{task_id}[{idx}].{key}: must be non-empty list-of-lists")
                continue
            row_len = len(grid[0])
            for r, row in enumerate(grid):
                if not isinstance(row, list):
                    errors.append(f"{task_id}[{idx}].{key}[{r}]: row must be list")
                    break
                if len(row) != row_len:
                    errors.append(f"{task_id}[{idx}].{key}[{r}]: jagged row (expected {row_len}, got {len(row)})")
                    break
                for c, v in enumerate(row):
                    if not isinstance(v, int):
                        errors.append(f"{task_id}[{idx}].{key}[{r},{c}]: value must be int, got {type(v).__name__}({v!r})")
                        break
                    if not (0 <= v <= 9):
                        errors.append(f"{task_id}[{idx}].{key}[{r},{c}]: value {v} out of range 0-9")
                        break

# ── 6. Report ────────────────────────────────────────────────────────────────
print(f"Tasks: {len(sub)}")
print(f"Total output entries: {total_entries}")

if errors:
    print(f"\nFAIL — {len(errors)} error(s):")
    for e in errors[:20]:
        print(f"  {e}")
    if len(errors) > 20:
        print(f"  ... and {len(errors)-20} more")
    sys.exit(1)
else:
    print(f"Valid: {SUB_PATH}")
